In [ ]:

import sys
import torch
sys.path.append('..')
from source.mamba import Mamba


batch_size = 2
seq_len = 10
d_model = 64
d_state = 16
n_layers = 2
num_classes = 2

learning_rate=0.005
vocab_size = 20000
epochs=5
device = "cuda" if torch.cuda.is_available() else "cpu"
path_model = "../checkpoints/mamba_cls.pth"


# Create model
model = Mamba(
	d_model=d_model,
	n_layers=n_layers,
	d_state=d_state,
	expand_factor=2,
)

x = torch.randn(batch_size, seq_len, d_model)
    
print(f"Input shape: {x.shape}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# Forward pass
with torch.no_grad():
	output = model(x)

print(f"Output shape: {output.shape}")

In [ ]:
import os
import re
import string
from collections import Counter
from typing import List, Tuple

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader


# =========================
# Dataset class
# =========================
class TextDataset(Dataset):
    def __init__(self, texts: List[str], labels: List[int]):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return {
            "sentence": self.texts[idx],
            "label": self.labels[idx],
        }


# =========================
# Processing class
# =========================
class ProcessingData:
    def __init__(
        self,
        data_dir: str,
        seq_length: int = 128,
        min_freq: int = 3,
        max_vocab_size: int = 10_000,
        batch_size: int = 32,
    ):
        self.data_dir = data_dir
        self.seq_length = seq_length
        self.min_freq = min_freq
        self.max_vocab_size = max_vocab_size
        self.batch_size = batch_size

        # Load CSV
        self.df_train = pd.read_csv(os.path.join(data_dir, "train.csv"))
        self.df_val = pd.read_csv(os.path.join(data_dir, "val.csv"))
        self.df_test = pd.read_csv(os.path.join(data_dir, "test.csv"))

        # Preprocess text
        self.df_train["sentence"] = self.df_train["sentence"].apply(self._preprocess_text)
        self.df_val["sentence"]   = self.df_val["sentence"].apply(self._preprocess_text)
        self.df_test["sentence"]  = self.df_test["sentence"].apply(self._preprocess_text)

        # Build vocab
        self.vocab = self._build_vocab(self.df_train["sentence"].tolist())
        self.pad_id = self.vocab["<pad>"]
        self.unk_id = self.vocab["<unk>"]
        self.eos_id = self.vocab["</s>"]

        # Build datasets
        self.train_data = TextDataset(
            self.df_train["sentence"].tolist(),
            self.df_train["label"].tolist(),
        )
        self.val_data = TextDataset(
            self.df_val["sentence"].tolist(),
            self.df_val["label"].tolist(),
        )
        self.test_data = TextDataset(
            self.df_test["sentence"].tolist(),
            self.df_test["label"].tolist(),
        )

    # -------------------------
    # Text preprocessing
    # -------------------------
    def _preprocess_text(self, text: str) -> str:
        if not isinstance(text, str):
            return ""

        # remove URLs
        text = re.sub(r"https?://\S+|www\.\S+", " ", text)

        # remove HTML tags
        text = re.sub(r"<[^<>]+>", " ", text)

        # remove punctuation & digits
        text = text.translate(str.maketrans("", "", string.punctuation + string.digits))

        # remove emojis
        emoji_pattern = re.compile(
            "["
            u"\U0001F600-\U0001F64F"
            u"\U0001F300-\U0001F5FF"
            u"\U0001F680-\U0001F6FF"
            u"\U0001F1E0-\U0001F1FF"
            u"\u200d"
            u"\u2640-\u2642"
            "]+",
            flags=re.UNICODE,
        )
        text = emoji_pattern.sub(" ", text)

        # normalize whitespace & lowercase
        text = " ".join(text.split()).lower()
        return text

    # -------------------------
    # Vocabulary
    # -------------------------
    def _build_vocab(self, texts: List[str]) -> dict:
        counter = Counter()
        for text in texts:
            counter.update(text.split())

        vocab = {
            "<pad>": 0,
            "</s>": 1,
            "<unk>": 2,
        }

        for word, freq in counter.items():
            if freq >= self.min_freq and len(vocab) < self.max_vocab_size:
                vocab[word] = len(vocab)

        return vocab

    # -------------------------
    # Tokenize + pad
    # -------------------------
    def _encode(self, text: str) -> List[int]:
        tokens = [self.vocab.get(tok, self.unk_id) for tok in text.split()]
        tokens = tokens[: self.seq_length - 1]          # reserve space for </s>
        tokens.append(self.eos_id)

        pad_size = self.seq_length - len(tokens)
        tokens.extend([self.pad_id] * pad_size)
        return tokens

    # -------------------------
    # Collate function
    # -------------------------
    def _collate_batch(self, batch):
        texts = [self._encode(sample["sentence"]) for sample in batch]
        labels = [sample["label"] for sample in batch]

        input_ids = torch.tensor(texts, dtype=torch.long)
        label_ids = torch.tensor(labels, dtype=torch.long)
        return input_ids, label_ids

    # -------------------------
    # Dataloaders
    # -------------------------
    def create_dataloaders(self) -> Tuple[DataLoader, DataLoader, DataLoader]:
        train_loader = DataLoader(
            self.train_data,
            batch_size=self.batch_size,
            shuffle=True,
            collate_fn=self._collate_batch,
        )

        val_loader = DataLoader(
            self.val_data,
            batch_size=self.batch_size,
            shuffle=False,
            collate_fn=self._collate_batch,
        )

        test_loader = DataLoader(
            self.test_data,
            batch_size=self.batch_size,
            shuffle=False,
            collate_fn=self._collate_batch,
        )

        return train_loader, val_loader, test_loader

processor = ProcessingData(data_dir="../data")
train_loader, val_loader, test_loader = processor.create_dataloaders()
print(len(train_loader))
print(len(val_loader))
print(len(test_loader))

In [ ]:
import time 
import numpy as np
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    precision_score,
    confusion_matrix,
    roc_auc_score
)

def train_epoch(model: nn.Module, 
                optimizer: torch.optim.AdamW, 
                criterion: nn.CrossEntropyLoss, 
                train_loader: DataLoader, 
                device: str, 
                epoch: int, 
                log_interval: int = 50):
    
    model.train()
    train_acc , train_loss = 0.0 , 0.0
    total = 0.0

    for idx, (sentence, label) in enumerate(train_loader):
        sentence = sentence.to(device)
        labels = label.to(device)
        optimizer.zero_grad()

        outputs = model(sentence) # tính outputs của model
        loss = criterion(outputs, labels) # tính loss
        train_loss += loss.item()

        #backward
        loss.backward()
        optimizer.step()

        train_acc += (torch.argmax(outputs, 1) == labels).sum().item()
        total += len(labels)

        if idx % log_interval == 0 and idx > 0:
            print(
                "| Epoch {:3d} | {}/{} batches | accuracy {:4.2f}".format(
                    epoch, idx, len(train_loader), train_acc / total
                )
            )
    epoch_acc = train_acc / total
    epoch_loss = train_loss / len(train_loader)
    return epoch_acc, epoch_loss


def evaluate_epoch(model: nn.Module, 
                   criterion: nn.CrossEntropyLoss, 
                   val_loader: DataLoader, 
                   device: str):
    val_acc, val_loss = 0.0, 0.0
    total = 0.0

    model.eval()
    with torch.no_grad():
        for idx, (sentences, labels) in enumerate(val_loader):
            sentences = sentences.to(device)
            labels = labels.to(device)

            outputs = model(sentences)
            loss = criterion(outputs, labels)

            val_acc += (torch.argmax(outputs, 1) == labels).sum().item()
            val_loss += loss.item()
            total += len(labels)

    epoch_acc = val_acc / total
    epoch_loss = val_loss / len(val_loader)
    return epoch_acc, epoch_loss


def fit(model: nn.Module, 
				optimizer: torch.optim.AdamW, 
				criterion: nn.CrossEntropyLoss, 
				train_loader: DataLoader, 
				val_loader: DataLoader, 
				num_epochs: int,
				path_model: str, 
				device: str):

    train_accs, train_losses = [], []
    eval_accs, eval_losses = [], []
    best_loss_eval = 100
    times = []
    print("Start training Mamba on Classification task.")
    
    for epoch in range(1, num_epochs+1):
        epoch_start_time = time.time()

        # trainning
        train_acc, train_loss = train_epoch(model, optimizer, criterion, train_loader, device, epoch)
        train_accs.append(train_acc)
        train_losses.append(train_loss)

        # evaluate
        eval_acc, eval_loss = evaluate_epoch(model, criterion, val_loader, device)
        eval_accs.append(eval_acc)
        eval_losses.append(eval_loss)

        # save model
        if eval_loss < best_loss_eval:
            torch.save(model.state_dict(), path_model)
            best_loss_eval = eval_loss

        times.append(time.time() - epoch_start_time)
        print("-" * 60)
        print(
            "| end of epoch {:3d} | Time {:5.2f}s | Train Acc {:8.3f} | Train Loss {:8.3f} "
            "| Val Acc {:8.3f} | Val Loss {:8.3f} ".format(
                epoch, time.time() - epoch_start_time, train_acc, train_loss, eval_acc, eval_loss
            )
        )
        print("-" * 60)

    model.load_state_dict(torch.load(f=path_model, 
                                     map_location=torch.device(device=device)))
    model.eval()
    metrics = {
        'train_accuracies': train_accs,
        'train_losses': train_losses,
        'valid_accuracies': eval_accs,
        'valid_losses': eval_losses,
        'time': times
    }
    return model, metrics


def evaluate(model: nn.Module,
            data_loader: DataLoader,
            device: str = 'cuda',
            task_type: str = 'binary') -> dict:
    """
    Evaluate model performance with multiple metrics
    
    Args:
        model: PyTorch model
        data_loader: PyTorch DataLoader containing validation/test data
        device: Device to run evaluation on ('cuda' or 'cpu')
        task_type: Type of classification task ('binary' or 'multi')
    
    Returns:
        Dictionary containing evaluation metrics
    """
    # Validate arguments
    if task_type not in ['binary', 'multi']:
        raise ValueError("task_type must be either 'binary' or 'multi'")

    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for batch in data_loader:
            inputs, labels = batch
            inputs = inputs.to(device)
            labels = labels.to(device)

            # Forward pass
            logits = model(inputs)

            # Get predictions and probabilities
            if task_type == 'binary':
                probs = logits[:, 1].sigmoid()
                preds = (probs > 0.5).int()
                all_probs.append(probs.cpu().numpy())
            else:
                preds = logits.argmax(dim=1)

            # Store batch results
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    # Concatenate all batches
    all_labels = np.concatenate(all_labels)
    all_preds = np.concatenate(all_preds)

    # Calculate metrics
    metrics = {
        'accuracy': accuracy_score(all_labels, all_preds),
        'precision': precision_score(
            all_labels, 
            all_preds,
            average='binary' if task_type == 'binary' else 'macro'
        ),
        'recall': recall_score(
            all_labels, 
            all_preds, 
            average='binary' if task_type == 'binary' else 'macro'
        ),
        'f1': f1_score(
            all_labels, 
            all_preds,
            average='binary' if task_type == 'binary' else 'macro'
        ),
        'confusion_matrix': confusion_matrix(all_labels, all_preds)
    }

    # Add ROC-AUC for binary classification
    if task_type == 'binary':
        all_probs = np.concatenate(all_probs)
        metrics['roc_auc'] = roc_auc_score(all_labels, all_probs)

    return metrics

In [ ]:
import torch.nn as nn 


class MambaClassifier(nn.Module):
	def __init__(self, vocab_size, d_model, n_layers, num_classes):
		super().__init__()
		self.embedding = nn.Embedding(vocab_size, d_model)
		self.mamba = Mamba(
				d_model=d_model,
				n_layers=n_layers,
				d_state=16,
				expand_factor=2
		)
		self.classifier = nn.Linear(d_model, num_classes)

	def forward(self, x):
			# x: (batch, seq_len)
			x = self.embedding(x)  # (batch, seq_len, d_model)
			x = self.mamba(x)      # (batch, seq_len, d_model)
			# Pool: take last token representation
			x = x[:, -1, :]        # (batch, d_model)
			logits = self.classifier(x)  # (batch, num_classes)
			return logits



In [ ]:
classifier = MambaClassifier(vocab_size=vocab_size, d_model=d_model, 
                        		n_layers=n_layers, num_classes=num_classes)
classifier = classifier.to(device)  # Move model to device

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(params=classifier.parameters(), lr=learning_rate)
model, metrics = fit(model=classifier, 
                     optimizer=optimizer, 
                     criterion=criterion, 
											train_loader=train_loader, 
                      val_loader=val_loader, num_epochs=epochs, 
											path_model=path_model, device=device)

In [ ]:
# Trace Input qua Model Mamba
## Phân tích chi tiết cách input được biến đổi qua từng layer của MambaClassifier

In [ ]:
# Lấy 1 batch sample từ train_loader
sample_input, sample_label = next(iter(train_loader))

print("="*80)
print("TRACE INPUT QUA TOÀN BỘ MODEL MAMBA")
print("="*80)

# Chỉ lấy 1 sample đầu tiên để dễ quan sát
single_input = sample_input[0:1]  # (1, seq_len)
single_label = sample_label[0:1]  # (1,)

print(f"\n1. INPUT GỐC:")
print(f"   Shape: {single_input.shape}")
print(f"   Giải thích: (batch_size=1, seq_length={single_input.shape[1]})")
print(f"   Đây là chuỗi token IDs với độ dài {single_input.shape[1]}")
print(f"   Ví dụ 10 token đầu tiên: {single_input[0, :10].tolist()}")

# Chuyển sang device
single_input = single_input.to(device)

print("\n" + "="*80)
print("BẮT ĐẦU FORWARD PASS QUA MAMBACCLASSIFIER")
print("="*80)

# 1. Embedding Layer
embedded = classifier.embedding(single_input)
print(f"\n2. SAU EMBEDDING LAYER:")
print(f"   Shape: {embedded.shape}")
print(f"   Giải thích: (batch_size=1, seq_length={embedded.shape[1]}, d_model={embedded.shape[2]})")
print(f"   Mỗi token ID được chuyển thành vector {embedded.shape[2]} chiều")
print(f"   Từ {single_input.shape} → {embedded.shape}")

# 2. Qua Mamba Backbone
# Trace qua từng layer của Mamba
print(f"\n3. ĐI QUA MAMBA BACKBONE ({classifier.mamba.n_layers} layers):")
print(f"   Input vào Mamba: {embedded.shape}")

mamba_output = embedded
for layer_idx, mamba_layer in enumerate(classifier.mamba.layers):
    # Mỗi layer bao gồm: RMSNorm → MambaBlock → Residual
    print(f"\n   --- Layer {layer_idx + 1}/{classifier.mamba.n_layers} ---")
    
    # Normalize
    normalized = mamba_layer.norm(mamba_output)
    print(f"   Sau RMSNorm: {normalized.shape}")
    print(f"   (Chỉ normalize, không đổi shape)")
    
    # MambaBlock
    # Input projection
    x_proj = mamba_layer.mixer.in_proj(normalized)
    print(f"   Sau in_proj: {x_proj.shape}")
    print(f"   (Linear: {normalized.shape[2]} → {x_proj.shape[2]})")
    
    # Split thành x và z
    x, z = x_proj.chunk(2, dim=-1)
    print(f"   Split thành x: {x.shape}, z: {z.shape}")
    print(f"   (Chia đôi dimension cuối)")
    
    # Conv1d
    x_conv = mamba_layer.mixer.conv1d(x.transpose(1, 2)).transpose(1, 2)
    print(f"   Sau Conv1d: {x_conv.shape}")
    print(f"   (Conv1d với kernel_size={mamba_layer.mixer.conv1d.kernel_size[0]}, padding={mamba_layer.mixer.conv1d.padding[0]})")
    
    # Activation
    x_activated = torch.nn.functional.silu(x_conv)
    print(f"   Sau SiLU activation: {x_activated.shape}")
    
    # SSM projection
    x_ssm = mamba_layer.mixer.x_proj(x_activated)
    print(f"   Sau SSM projection: {x_ssm.shape}")
    
    # Output projection  
    block_output = mamba_layer.mixer.out_proj(x_activated)
    print(f"   Sau out_proj: {block_output.shape}")
    
    # Residual connection
    mamba_output = mamba_output + block_output
    print(f"   Sau residual connection: {mamba_output.shape}")
    print(f"   (Cộng với input ban đầu của layer)")

# Final norm
final_normed = classifier.mamba.norm(mamba_output)
print(f"\n4. SAU FINAL RMSNorm:")
print(f"   Shape: {final_normed.shape}")
print(f"   (Normalize lần cuối, giữ nguyên shape)")

# 3. Pooling - Lấy token cuối cùng
pooled = final_normed[:, -1, :]
print(f"\n5. SAU POOLING (lấy token cuối):")
print(f"   Shape: {pooled.shape}")
print(f"   Giải thích: (batch_size=1, d_model={pooled.shape[1]})")
print(f"   Chỉ lấy output của token cuối cùng (index -1)")
print(f"   Từ {final_normed.shape} → {pooled.shape}")

# 4. Classification Head
logits = classifier.classifier(pooled)
print(f"\n6. SAU CLASSIFIER (Linear Layer cuối):")
print(f"   Shape: {logits.shape}")
print(f"   Giải thích: (batch_size=1, num_classes={logits.shape[1]})")
print(f"   Linear projection: d_model={pooled.shape[1]} → num_classes={logits.shape[1]}")

# Prediction
probs = torch.softmax(logits, dim=-1)
predicted_class = torch.argmax(logits, dim=-1)

print(f"\n7. OUTPUT CUỐI CÙNG:")
print(f"   Logits: {logits.shape}")
print(f"   Probabilities: {probs}")
print(f"   Predicted class: {predicted_class.item()}")
print(f"   True label: {single_label.item()}")

print("\n" + "="*80)
print("TỔNG KẾT BIẾN ĐỔI SHAPE:")
print("="*80)
print(f"Input tokens:    {single_input.shape} → (batch=1, seq_len={single_input.shape[1]})")
print(f"Embedding:       {embedded.shape} → (batch=1, seq_len={embedded.shape[1]}, d_model={embedded.shape[2]})")
print(f"Mamba layers:    {final_normed.shape} → (giữ nguyên qua {classifier.mamba.n_layers} layers)")
print(f"Pooling:         {pooled.shape} → (batch=1, d_model={pooled.shape[1]})")
print(f"Classification:  {logits.shape} → (batch=1, num_classes={logits.shape[1]})")
print("="*80)